# Bahraini Currency Recognition with Deep Learning

The aim is to classify a picture of Bahraini currency into its correct class.

# 1. Data Collection

The supplied dataset contains Bahraini currency photographed with different backgrounds, lighting conditions, angles and distances. It contains paper notes and optional coins.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils import compute_class_weight
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import set_random_seed

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
seed = 100

random.seed(seed)
np.random.seed(seed)
set_random_seed(seed)

In [ ]:
data_path = "/content/Currency"
image_size = 64

class_names = ["0.05", "0.100", "0.25", "0.5 BD", "0.50",
               "1BD", "5 BD", "10 BD", "20 BD"]
class_names

In [ ]:
images = []
labels = []

for label, class_name in enumerate(class_names):
    for folder_name in ["Train", "Test"]:
        folder = os.path.join(data_path, class_name, folder_name)

        for file_name in os.listdir(folder):
            image_path = os.path.join(folder, file_name)

            if os.path.isfile(image_path):
                image = Image.open(image_path).convert("RGB")
                image = image.resize((image_size, image_size))

                images.append(np.array(image))
                labels.append(label)

In [ ]:
X = np.array(images)
y = np.array(labels)

X.shape, y.shape

In [ ]:
image_counts = pd.Series(y).value_counts().sort_index()
image_counts.index = class_names
image_counts

The supplied dataset contains 9 classes and 474 images in total. The lab recommends at least 10 classes, 50 images for every class, and 300-500 images in total. The output above shows the actual number available in each class. One additional class and extra original photos for classes below 50 images are needed to meet every dataset recommendation exactly.

# 2. Preprocessing

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
# A genuine validation split is carved out here BEFORE augmentation.
# This matters: the augmentation step below duplicates every training image
# with a small rotation/zoom/brightness change. If an augmented copy is
# created first and `validation_split` is used inside model.fit (as before),
# Keras takes the LAST 20% of the array as validation data - which, after
# concatenation, is entirely augmented images whose original, unaugmented
# counterpart is still sitting in the training set. The model would then be
# "validated" on near-duplicates of images it just trained on, which makes
# val_loss/val_acc and EarlyStopping look better than the model actually
# generalizes - a likely reason the notebook metrics looked fine while real
# phone photos in the Streamlit app were not classified correctly.
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train)

x_train.shape, x_val.shape, x_test.shape


## Data Augmentation

In [ ]:
augmentation = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2])

In [ ]:
new_images, new_labels = next(
    augmentation.flow(
        x_train,
        y_train,
        batch_size=len(x_train),
        shuffle=False,
        seed=seed))

x_train = np.concatenate([x_train, new_images])
y_train = np.concatenate([y_train, new_labels])

Augmentation creates rotated, zoomed, flipped and brighter or darker training images. It helps the model recognize currency photographed from different angles, distances and lighting conditions.

## Normalize Pixel Values

In [ ]:
x_train_scaled = x_train / 255
x_val_scaled = x_val / 255
x_test_scaled = x_test / 255


# 3. Model Training (Classification)

In [ ]:
# Step 1: Import was completed above

# Step 2: Define
model = Sequential()
model.add(Flatten(input_shape=(image_size, image_size, 3)))
model.add(Dense(500, activation="relu"))
model.add(Dense(200, activation="relu"))
model.add(Dense(100, activation="relu"))
model.add(Dense(len(class_names), activation="softmax"))

model.summary()

In [ ]:
# Step 3: Compile
model.compile(
    loss="sparse_categorical_crossentropy",
    metrics=["acc"])

## Class Weights

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

class_weight_dictionary = dict(enumerate(class_weights))
class_weight_dictionary

## Early Stopping

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5)

In [ ]:
# Step 4: Fit
history = model.fit(
    x_train_scaled,
    y_train,
    validation_data=(x_val_scaled, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    class_weight=class_weight_dictionary)


## Training History

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.tail()

In [ ]:
history_df[["acc", "val_acc"]].plot()
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
history_df[["loss", "val_loss"]].plot()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

# 4. Model Evaluation

In [ ]:
# Step 5: Prediction
y_pred = model.predict(x_test_scaled)
y_pred_label = y_pred.argmax(axis=1)

## Accuracy

In [ ]:
accuracy_score(y_test, y_pred_label)

## Precision, Recall and Per-Class Performance

In [ ]:
print(classification_report(
    y_test,
    y_pred_label,
    labels=range(len(class_names)),
    target_names=class_names,
    zero_division=0))

The classification report displays precision, recall and F1-score separately for every currency class.

## Confusion Matrix

In [ ]:
conf_matrix = confusion_matrix(
    y_test,
    y_pred_label,
    labels=range(len(class_names)))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    conf_matrix,
    annot=True,
    cmap="Blues",
    fmt="g",
    xticklabels=class_names,
    yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Error Analysis

In [ ]:
idx = np.where(y_test != y_pred_label)[0]
idx

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(min(8, len(idx))):
    plt.subplot(2, 4, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[idx[i]])
    plt.xlabel(
        f"Actual: {class_names[y_test[idx[i]]]}\n"
        f"Pred: {class_names[y_pred_label[idx[i]]]}")

plt.tight_layout()
plt.show()

# 5. Model Deployment (Streamlit)

In [ ]:
model.save("currency_model.keras")

The saved model is used by `app.py`. The Streamlit app accepts an uploaded mobile-phone photo or a live-camera photo, resizes it to 64 x 64, and displays the predicted currency.

# 6. Key Questions

In [ ]:
confused_matrix = conf_matrix.copy()
np.fill_diagonal(confused_matrix, 0)

actual_class, predicted_class = np.unravel_index(
    confused_matrix.argmax(), confused_matrix.shape)

print("Most confused actual currency:", class_names[actual_class])
print("Most common wrong prediction:", class_names[predicted_class])

1. **Which currency is most confusing?** The output above identifies the actual currency that produced the most common wrong prediction.
2. **Are similar-looking notes misclassified?** Compare the two currencies printed above and review their incorrect images in the error-analysis section. Similar colours, shapes or designs can make them harder to separate.
3. **Does background affect performance?** The incorrect images should be checked for hands, patterned surfaces, shadows and distant currency. If these backgrounds appear often among the errors, the background is affecting performance.

# 7. Real-World Testing

Use the Streamlit app to test the model with:

- A live-camera photo
- A mobile-phone photo
- A photo with a new background

## Discussion

- **False positives:** When the model incorrectly predicts a class, that result is a false positive for the predicted currency.
- **Missed detections:** In this classification task, an incorrect result means the real currency class was missed. Classification also cannot locate several currencies separately in one image.
- **Speed vs accuracy:** Resizing to 64 x 64 makes training and prediction faster, but a larger image size would keep more detail and require more processing.

# 8. Final Reflection

1. **Why did classification work well initially?** It worked well because each image mainly contained one currency and each image had one clear class label.
2. **What problems did object detection solve?** Object detection would locate each currency separately and could handle more than one note or coin in the same image.
3. **Which approach is more suitable for real-world deployment?** Classification is suitable when the user shows one currency at a time. Object detection is more suitable when several currencies may appear together.
4. **How would you improve the system further?** Collect at least 50 different original images for every class, add more unseen backgrounds and lighting conditions, and test the model using more new mobile-phone photos.